# Atividade: construindo a MLP

Na aula, a gente montou uma rede neural peça por peça: os pesos, o forward, a perda, o backpropagation e o passo que ajusta os pesos. No fim, juntamos tudo em poucas linhas de PyTorch.

Neste notebook você vai reconstruir esse código. Ele já está quase todo escrito, como nos slides, mas algumas linhas foram apagadas. Cada lacuna aparece como `...` e vem com um comentário dizendo o que falta. Os números ①, ②, ③… são os mesmos marcadores dos slides.

Depois de cada lacuna há uma célula de **conferência**. Ela compara o seu resultado com os números que vimos em aula. Se a conferência falhar, volte à lacuna antes de seguir: as partes seguintes dependem dela.

A rede é a mesma da aula: **2 entradas, 4 neurônios ocultos com tanh e 3 saídas**, treinada no Iris com o comprimento e a largura da sépala.

> **Antes de editar:** use **Arquivo → Salvar uma cópia no Drive** e trabalhe na sua cópia.

## 1. Os dados

Esta parte já está pronta. Usamos as duas medidas da sépala, como na aula, e separamos 120 flores para treino e 30 para teste. As medidas são levadas para a faixa de 0 a 1 com o mínimo e o máximo **do treino**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

SPECIES = ['setosa', 'versicolor', 'virginica']

iris = load_iris(as_frame=True)
X = iris.data[['sepal length (cm)', 'sepal width (cm)']]
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=30, stratify=y, random_state=0)

lo, hi = X_train.min(), X_train.max()
X_train = (X_train - lo) / (hi - lo)
X_test = (X_test - lo) / (hi - lo)

print(f'{len(X_train)} flores de treino, {len(X_test)} de teste')
X_train.describe().round(2)

In [ ]:
for k, name in enumerate(SPECIES):
    sel = y_train == k
    plt.scatter(X_train[sel].iloc[:, 0], X_train[sel].iloc[:, 1], label=name, s=25,
                color=plt.cm.viridis(k / 2))
plt.xlabel('comprimento da sépala (0 a 1)')
plt.ylabel('largura da sépala (0 a 1)')
plt.legend()
plt.show()

**Antes de programar, responda:** olhando o gráfico, alguma espécie é fácil de separar das outras? Alguma fica misturada? Uma reta só daria conta de separar as três?

**Minha resposta:** ...

## 2. Os pesos da rede

Cada camada tem uma matriz de pesos e um vetor de vieses:

- `W1` liga as **2 entradas** aos **4 neurônios ocultos**, e `b1` tem um viés por neurônio oculto;
- `W2` liga os **4 ocultos** às **3 saídas**, e `b2` tem um viés por saída.

Na matriz, a linha é o neurônio de onde o sinal sai, e a coluna é o neurônio aonde ele chega. Assim, `W1[0, 0]` é o peso **w₁₁** da aula, da entrada 1 para o oculto 1.

`nn.Parameter` avisa ao PyTorch que aquele tensor é um peso: ele vai calcular a derivada do erro em relação a ele e o otimizador vai ajustá-lo. Os pesos começam sorteados com `torch.randn(linhas, colunas)` e os vieses começam em zero com `torch.zeros(tamanho)`.

Complete as lacunas ① e ②. A ③ e a ④ ficam para as próximas seções.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in=2, n_hid=4, n_out=3, lr=1e-3, epochs=200):
        super().__init__()
        self.epochs = epochs
        self.W1 = ...                                        # ① entrada → oculta: matriz n_in × n_hid
        self.b1 = nn.Parameter(torch.zeros(n_hid))
        self.W2 = ...                                        # ② oculta → saída: matriz n_hid × n_out
        self.b2 = ...                                        # ② um viés por saída
        self.loss_fn = ...                                   # ③ (seção 4)
        self.opt = torch.optim.Adam(self.parameters(), lr=lr)

    def forward(self, x):                                    # ④ (seção 3)
        ...

In [ ]:
# Conferência dos pesos.
torch.manual_seed(0)
mlp = MLP()
for name, p in mlp.named_parameters():
    print(f'{name}: {tuple(p.shape)}')

n_pesos = mlp.W1.numel() + mlp.W2.numel()
n_vieses = mlp.b1.numel() + mlp.b2.numel()
print(f'{n_pesos} pesos e {n_vieses} vieses')  # esperado: 20 pesos e 7 vieses
assert n_pesos == 20 and n_vieses == 7

## 3. O forward

O forward é a conta que leva a flor da entrada até a saída, como fizemos no quadro:

1. a camada oculta soma as entradas com os pesos e aplica a tanh: `z = tanh(x · W1 + b1)`;
2. a camada de saída soma os ocultos com os pesos, **sem** ativação: `s = z · W2 + b2`.

O resultado `s` são os **logits**, um número por espécie. Em PyTorch, `@` é a multiplicação de matrizes e `torch.tanh` aplica a tanh elemento a elemento.

Complete o `forward` (lacuna ④) na célula da classe e rode-a de novo.

Para conferir, vamos colocar na rede os **mesmos pesos da aula** e passar o **Exemplo 1**, a versicolor com sépala de 6,6 cm por 3,0 cm. Com a normalização usada na aula, ela entra na rede como `x = (0,68 ; 0,45)`.

In [ ]:
# Os pesos da aula (os mesmos dos slides do forward e do backprop).
AULA = {
    'W1': [[-1.735176, 2.606524, -1.481376, -2.786480],
           [-0.418315, -1.932969, 0.056123, 2.058232]],
    'b1': [1.276070, 0.109296, 0.774261, -0.115400],
    'W2': [[0.605910, 0.964865, -1.585338],
           [-2.776901, 1.433143, 1.349227],
           [0.896182, 0.400764, -1.310416],
           [2.993653, -1.567219, -1.408736]],
    'b2': [0.206727, -0.068409, 0.520939],
}
x_ex1 = torch.tensor([[0.676471, 0.454545]])
y_ex1 = torch.tensor([1])  # versicolor


def mlp_da_aula():
    rede = MLP()
    with torch.no_grad():
        for name, values in AULA.items():
            getattr(rede, name).copy_(torch.tensor(values))
    return rede

In [ ]:
# Conferência do forward.
rede = mlp_da_aula()
s = rede(x_ex1)
print('logits s =', s.detach().numpy().round(2))         # esperado: [[-4.49  2.09  3.05]]
assert torch.allclose(s, torch.tensor([[-4.49, 2.09, 3.05]]), atol=0.01)

Os logits ainda não são probabilidades. A **softmax** os transforma em números entre 0 e 1 que somam 1, e o **argmax** escolhe a espécie com o maior valor.

In [ ]:
p = torch.softmax(s, dim=-1)
print('probabilidades =', p.detach().numpy().round(3))
print('a rede responde:', SPECIES[torch.argmax(p).item()], '| a resposta certa é:', SPECIES[y_ex1.item()])

**Responda:**

1. Com os pesos da aula, a rede acerta o Exemplo 1? Com quanta certeza?
2. Por que a camada de saída não leva tanh?

**Minhas respostas:** ...

## 4. A perda

Para treinar, precisamos de um número que diga **o quanto a rede errou**. Em classificação, usamos a **entropia cruzada**: aplicamos a softmax aos logits e olhamos a probabilidade que a rede deu à resposta certa. O erro é

$$E = -\log p_{\text{certa}}$$

Se a rede dá probabilidade 1 à espécie certa, o erro é zero. Quanto menor a probabilidade, maior o erro.

Primeiro, escreva a entropia cruzada à mão. Depois complete a lacuna ③ na classe com `nn.CrossEntropyLoss()`, que faz a mesma conta e já aplica a softmax por dentro. É por isso que o `forward` devolve logits, e não probabilidades.

In [ ]:
def cross_entropy(s, y):
    """s: logits (1, 3); y: índice da espécie certa (1,). Devolve -log da probabilidade certa."""
    # 1. aplique a softmax em s
    # 2. pegue a probabilidade da espécie y
    # 3. devolva o -log dela
    ...

In [ ]:
# Conferência da perda. Rode a célula da classe de novo depois de preencher a ③.
rede = mlp_da_aula()
s = rede(x_ex1)
a_mao = cross_entropy(s, y_ex1).item()
pytorch = rede.loss_fn(s, y_ex1).item()
print(f'à mão: {a_mao:.3f} | PyTorch: {pytorch:.3f}')   # esperado: 1.289 nos dois
assert abs(a_mao - 1.289) < 0.01 and abs(pytorch - 1.289) < 0.01

## 5. O backpropagation, por dentro

No treino, quem calcula as derivadas é o PyTorch: basta chamar `.backward()` no erro, e cada peso ganha um `.grad`, que é **∂E/∂w** daquele peso. Antes de confiar nele, vamos conferir uma dessas derivadas com a conta que fizemos em aula.

Nos slides do backprop usamos o MSE, para facilitar a visualização:

$$E = \tfrac{1}{2} \sum_k (s_k - y_k)^2$$

com `y` em one-hot: a versicolor vira `(0, 1, 0)`. Na aula, seguimos o caminho que passa por **s₁** e, no fim, somamos os três caminhos para chegar a **∂E/∂w₁₁ ≈ −4,37**.

Complete as duas lacunas: o erro `E` e a chamada que dispara o backpropagation.

In [ ]:
rede = mlp_da_aula()
s = rede(x_ex1)
y_onehot = torch.tensor([[0., 1., 0.]])

E = ...      # ½ Σ (s − y)²
...          # backpropagation: preenche o .grad de todos os pesos

print(f'E = {E.item():.2f}')                                # esperado: 15.34
print(f'∂E/∂w11 pelo PyTorch = {rede.W1.grad[0, 0].item():.2f}')  # esperado: -4.37

Agora refaça a mesma derivada **à mão**, com a regra da cadeia da aula. Cada fator é um elo do dominó:

$$\frac{\partial E}{\partial w_{11}} = \underbrace{\sum_k (s_k - y_k)\, v_{1k}}_{\text{as três saídas}} \cdot \underbrace{(1 - z_1^2)}_{\text{tanh}'} \cdot \underbrace{x_1}_{\text{entrada}}$$

Os `v₁ₖ` são os pesos que saem do oculto 1, ou seja, a primeira linha de `W2`.

In [ ]:
with torch.no_grad():
    z = torch.tanh(x_ex1 @ rede.W1 + rede.b1)   # os 4 ocultos
    s = z @ rede.W2 + rede.b2                   # as 3 saídas
    x1, z1 = x_ex1[0, 0], z[0, 0]

    pelas_saidas = ...   # Σ_k (s_k − y_k) · v_1k
    derivada_tanh = ...  # 1 − z1²
    a_mao = ...          # junte os três elos

print(f'∂E/∂w11 à mão = {a_mao.item():.2f}')
assert torch.isclose(a_mao, rede.W1.grad[0, 0], atol=1e-4)

**Responda:**

1. O sinal de ∂E/∂w₁₁ é negativo. Para diminuir o erro, w₁₁ deve aumentar ou diminuir?
2. Olhe `rede.W1.grad` inteiro. Qual peso da primeira camada tem a maior "culpa" pelo erro no Exemplo 1?

**Minhas respostas:** ...

## 6. O treino

Agora juntamos tudo. Para cada flor do treino, um passo de treino tem quatro tempos, os mesmos do slide:

1. **zerar** as derivadas do passo anterior, senão o PyTorch soma as novas às antigas;
2. **forward** e **erro**;
3. **backpropagation**: calcular ∂E/∂w de todos os pesos;
4. **ajustar** os pesos, um passo contra a culpa de cada um.

Uma **época** é uma passada por todas as flores do treino. Para acompanhar o treino, guardamos o erro médio de cada época em `history`.

A função abaixo vira um método da classe `MLP` na última linha: escrevê-la fora da classe só serve para a gente montar o código em partes.

In [ ]:
def fit(self, X, y):
    X = torch.tensor(X.values, dtype=torch.float32)
    y = torch.tensor(y.values, dtype=torch.long)
    self.history = []
    for epoch in range(self.epochs):
        total = 0.0
        for xi, yi in zip(X, y):
            ...                                          # ① zera as derivadas
            s = ...                                      # ② forward de uma flor só
            loss = self.loss_fn(s, yi.unsqueeze(0))      # ② o erro
            ...                                          # ③ backpropagation
            ...                                          # ④ ajusta os pesos
            total += loss.item()
        self.history.append(total / len(X))
    return self


MLP.fit = fit

Dica para o ②: o `forward` espera uma **tabela** de flores, com uma linha por flor. `xi` é uma flor só, com formato `(2,)`; `xi.unsqueeze(0)` a transforma em uma tabela de uma linha, com formato `(1, 2)`.

O treino leva cerca de um minuto.

In [ ]:
torch.manual_seed(0)
mlp = MLP()
mlp.fit(X_train, y_train)

plt.plot(mlp.history)
plt.xlabel('época')
plt.ylabel('entropia cruzada média no treino')
plt.show()
print(f'erro na primeira época: {mlp.history[0]:.3f} | na última: {mlp.history[-1]:.3f}')

## 7. A inferência

Depois do treino, a rede fica **congelada**: na inferência não há erro, não há backpropagation e os pesos não mudam. Sobra só o forward e o argmax.

O decorador `@torch.no_grad()` avisa ao PyTorch que ele não precisa guardar as contas para calcular derivadas (lacuna ⑤).

In [ ]:
@torch.no_grad()                                         # ⑤ inferência: rede congelada
def predict(self, X):
    X = torch.tensor(X.values, dtype=torch.float32)
    ...                                                  # ⑤ forward e argmax, uma espécie por flor


MLP.predict = predict

In [ ]:
acc_train = accuracy_score(y_train.values, mlp.predict(X_train))
acc_test = accuracy_score(y_test.values, mlp.predict(X_test))
print(f'acurácia no treino: {acc_train:.2f} | no teste: {acc_test:.2f}')

O gráfico abaixo pinta cada ponto do plano com a espécie que a rede responderia. As linhas onde a cor muda são as **fronteiras de decisão**.

In [ ]:
def plot_fronteira(rede, title=''):
    g1, g2 = np.meshgrid(np.linspace(-0.1, 1.1, 300), np.linspace(-0.1, 1.1, 300))
    grade = pd.DataFrame({X_train.columns[0]: g1.ravel(), X_train.columns[1]: g2.ravel()})
    zona = rede.predict(grade).numpy().reshape(g1.shape)
    plt.contourf(g1, g2, zona, levels=[-0.5, 0.5, 1.5, 2.5], alpha=0.25, cmap='viridis')
    for k, name in enumerate(SPECIES):
        sel = y_train == k
        plt.scatter(X_train[sel].iloc[:, 0], X_train[sel].iloc[:, 1], label=name, s=20,
                    color=plt.cm.viridis(k / 2), edgecolor='white', linewidth=0.5)
    plt.xlabel('comprimento da sépala (0 a 1)')
    plt.ylabel('largura da sépala (0 a 1)')
    plt.title(title)
    plt.legend()
    plt.show()


plot_fronteira(mlp, 'MLP 2-4-3')

**Responda:**

1. Qual espécie a rede separa bem? Onde ela mais erra? Isso bate com o que você previu na seção 1?
2. Não espere 100% de acerto aqui. Por que só com a sépala isso é impossível, seja qual for a rede?
3. As fronteiras são retas ou curvas? Que parte da rede permite que elas se curvem?

**Minhas respostas:** ...

## 8. Mexendo na rede

Agora que a rede está pronta, mude uma peça de cada vez e observe o efeito. Fixe a semente antes de cada treino, para que as diferenças venham da mudança, e não do sorteio dos pesos iniciais.

### Quantos neurônios ocultos?

In [ ]:
for n_hid in [1, 2, 4, 16]:
    torch.manual_seed(0)
    rede = MLP(n_hid=n_hid).fit(X_train, y_train)
    acc = accuracy_score(y_test.values, rede.predict(X_test))
    plot_fronteira(rede, f'{n_hid} oculto(s): acurácia no teste {acc:.2f}')

### Sem a tanh

Crie uma rede igual à sua, mas sem a ativação na camada oculta. Basta herdar a classe `MLP` e reescrever só o `forward`.

In [ ]:
class MLPSemAtivacao(MLP):
    def forward(self, x):
        ...   # o mesmo forward, sem a tanh


torch.manual_seed(0)
linear = MLPSemAtivacao().fit(X_train, y_train)
acc = accuracy_score(y_test.values, linear.predict(X_test))
plot_fronteira(linear, f'sem tanh: acurácia no teste {acc:.2f}')

### A taxa de aprendizado

In [ ]:
for lr in [1e-4, 1e-3, 1e-1]:
    torch.manual_seed(0)
    rede = MLP(lr=lr).fit(X_train, y_train)
    plt.plot(rede.history, label=f'lr = {lr}')
plt.xlabel('época')
plt.ylabel('entropia cruzada média no treino')
plt.legend()
plt.show()

**Responda:**

1. Com 1 neurônio oculto, a rede consegue separar as três espécies? A partir de quantos neurônios a fronteira para de melhorar no teste?
2. Sem a tanh, como ficam as fronteiras? Por que duas camadas sem ativação equivalem a uma camada só?
3. Qual taxa de aprendizado demora demais para aprender? Qual oscila? O que isso lembra do gradiente descendente da semana passada?

**Minhas respostas:** ...

## 9. Fechamento

Responda com suas palavras:

1. Qual é a diferença entre o que a rede faz na **inferência** e o que ela faz no **treino**?
2. Por que é preciso zerar as derivadas (`zero_grad`) antes de cada passo? O que aconteceria sem isso?
3. O `.backward()` do PyTorch e a sua conta à mão deram o mesmo ∂E/∂w₁₁. O que o PyTorch fez para chegar lá sem você escrever a regra da cadeia?
4. Se você tivesse que explicar a um colega o que é uma época, o que diria?

**Minhas respostas:** ...

### Como entregar

Entregue o notebook executado, com as saídas visíveis.